In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
import faiss
import numpy as np
import os
from PIL import Image

# ── Step 1: Load CLIP model ──────────────────────────────
print("Loading CLIP model... (first time downloads ~600MB)")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()
print("✅ Model loaded!")

# ── Step 2: Function to get embedding from one image ─────
def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        output = model.get_image_features(**inputs)
        # Extract the tensor safely
        if isinstance(output, torch.Tensor):
            emb = output
        else:
            emb = getattr(output, "image_embeds", None) or getattr(output, "last_hidden_state", None)
        if emb is None:
            raise ValueError(f"Cannot find embedding tensor for {image_path}")
        emb = emb.cpu().numpy().flatten()
    return emb

# ── Step 3: Loop through all processed images ────────────
PROCESSED_FOLDER = r"D:\DataScience\data\processed_images"  # <-- full path recommended
EMBED_FOLDER = r"D:\DataScience\embeddings"
os.makedirs(EMBED_FOLDER, exist_ok=True)

embeddings = []
labels = []
image_paths = []

for product_name in os.listdir(PROCESSED_FOLDER):
    product_folder = os.path.join(PROCESSED_FOLDER, product_name)
    if not os.path.isdir(product_folder):
        continue

    for img_file in os.listdir(product_folder):
        if img_file.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(product_folder, img_file)
            emb = get_image_embedding(path)
            embeddings.append(emb)
            labels.append(product_name)
            image_paths.append(path)

    print(f"✅ Embedded: {product_name}")

# ── Step 4: Build and save FAISS index ───────────────────
embeddings_np = np.array(embeddings).astype("float32")
faiss.normalize_L2(embeddings_np)  # normalize for cosine similarity

dimension = embeddings_np.shape[1]  # 512 for CLIP
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_np)

faiss.write_index(index, os.path.join(EMBED_FOLDER, "product_index.faiss"))
np.save(os.path.join(EMBED_FOLDER, "labels.npy"), np.array(labels))
np.save(os.path.join(EMBED_FOLDER, "paths.npy"), np.array(image_paths))

print(f"\n🎉 Index built! Total images indexed: {index.ntotal}")